# Fruit Nutrition Knowledge Worker
A conversational AI grounded in a curated knowledge base of fruit nutritional data. Ask questions about vitamins, minerals, health benefits, and more across five fruits: Mango, Apple, Banana, Pawpaw, and Watermelon.

In [1]:
# imports

import os
import shutil
from pathlib import Path
from dotenv import load_dotenv
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
import gradio as gr

In [2]:
# constants

MODEL            = 'gpt-4o-mini'
EMBEDDING_MODEL  = 'text-embedding-3-small'
KNOWLEDGE_BASE   = 'knowledge_base/fruits'
CHROMA_PATH      = 'chroma_db'
COLLECTION_NAME  = 'fruits_nutrition'
CHUNK_SIZE       = 1000
CHUNK_OVERLAP    = 200

In [3]:
# set up environment

load_dotenv()

openai_api_key = os.getenv('OPENAI_API_KEY')

if not openai_api_key:
    print("❌ OpenAI API key not found")
else:
    print("✅ OpenAI API key found!")

✅ OpenAI API key found!


In [4]:
# load documents

loader = DirectoryLoader(
    KNOWLEDGE_BASE,
    glob="**/*.md",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"}
)

documents = loader.load()
print(f"📄 Loaded {len(documents)} documents")

📄 Loaded 5 documents


In [5]:
# chunk documents
splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP
)

chunks = splitter.split_documents(documents)
print(f"✂️  Split into {len(chunks)} chunks")

✂️  Split into 14 chunks


In [6]:
# embed and store in chroma 

embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)
if Path(CHROMA_PATH).exists():
    print("📦 Loading existing Chroma vector store...")
    vectorstore = Chroma(
        collection_name=COLLECTION_NAME,
        persist_directory=CHROMA_PATH,
        embedding_function=embeddings
    )
    if vectorstore._collection.count() < len(chunks):
        print("⚠️  Stale store detected, rebuilding...")
        shutil.rmtree(CHROMA_PATH)
        vectorstore = Chroma.from_documents(
            documents=chunks,
            embedding=embeddings,
            collection_name=COLLECTION_NAME,
            persist_directory=CHROMA_PATH
        )
else:
    print("🔄 Creating new Chroma vector store...")
    vectorstore = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        collection_name=COLLECTION_NAME,
        persist_directory=CHROMA_PATH
    )

retriever = vectorstore.as_retriever(search_kwargs={"k": 10})
print(f"✅ Vector store ready with {vectorstore._collection.count()} chunks")

🔄 Creating new Chroma vector store...
✅ Vector store ready with 14 chunks


In [7]:
# RAG function
system_prompt = """
You are a knowledgeable nutritionist assistant specialising in fruits.
Answer questions accurately using only the context provided from the knowledge base.
If the answer is not in the context, say so clearly.
Always be concise, friendly, and cite the fruit by name when relevant.
"""

llm = ChatOpenAI(model=MODEL, temperature=0.2, streaming=True)

def rag_response(question, history):
    question = question.strip()
    docs = retriever.invoke(question)

    context = "\n\n---\n\n".join([doc.page_content for doc in docs])
    messages = [{"role": "system", "content": system_prompt + f"\n\nContext:\n{context}"}]
    for msg in history:
        if msg["role"] != "assistant" or msg["content"] != welcome[0]["content"]:
            messages.append({"role": msg["role"], "content": msg["content"]})
    messages.append({"role": "user", "content": question})
    response = ""
    for chunk in llm.stream(messages):
        response += chunk.content
        yield history + [{"role": "assistant", "content": response}]

In [8]:
# Gradio UI
welcome = [{
    "role": "assistant",
    "content": (
        "🍎 Welcome to the Fruit Nutrition Assistant! "
        "I have detailed nutritional information on Mango, Apple, Banana, Pawpaw, and Watermelon. "
        "Ask me anything — vitamins, minerals, health benefits, calories, and more!"
    )
}]

with gr.Blocks() as ui:
    gr.Markdown("## 🍉 Fruit Nutrition Knowledge Worker")
    gr.Markdown("A conversational AI grounded in a curated fruit nutrition knowledge base. Ask questions and get accurate, context-backed answers.")
    chatbot = gr.Chatbot(value=welcome, height=500, type="messages")
    with gr.Row():
        question_box = gr.Textbox(
            lines=2,
            placeholder="e.g. What vitamins does a mango have?",
            label="Your Question"
        )
        submit_btn = gr.Button("Ask", variant="primary")

    def handle(question, history):
        return history + [{"role": "user", "content": question}]

    submit_btn.click(
        fn=lambda: gr.update(interactive=False),
        outputs=submit_btn
    ).then(
        fn=handle,
        inputs=[question_box, chatbot],
        outputs=chatbot
    ).then(
        fn=rag_response,
        inputs=[question_box, chatbot],
        outputs=chatbot
    ).then(
        fn=lambda: ("", gr.update(interactive=True)),
        outputs=[question_box, submit_btn]
    )

ui.launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
